# Initialization

In [0]:
from pyspark.sql import functions as F

# Define Ingestion Configuration

In [0]:
CATALOG = "humanoid_robotics"
SEEDS_PATH = f"/Volumes/{CATALOG}/landing/seeds"

SEED_CONFIG = [
    {"file": "company_seed.csv",  "table": f"{CATALOG}.bronze.company_seed"},
    {"file": "company_other.csv", "table": f"{CATALOG}.bronze.company_other"},
    {"file": "company_alias.csv", "table": f"{CATALOG}.bronze.company_alias"},
]

# Ingest Files into bronze table

In [0]:
for item in SEED_CONFIG:
    path = f"{SEEDS_PATH}/{item['file']}"
    print(f"Loading {path} → {item['table']}")

    df = (spark.read
            .option("header", True)
            .option("multiLine", True)
            .option("escape", '"')
            .csv(path)
            .select("*", F.col("_metadata.file_path").alias("source_file"))
        )
    df = df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])

    df = df.withColumn("ingested_at", F.current_timestamp())

    (df.write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(item["table"]))

    print(f"  → {df.count()} rows")

In [0]:
%sql

select * from humanoid_robotics.bronze.company_seed